In [1]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from textblob import TextBlob
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# ---------------------------------------------------------
# SETUP: Ensure VADER lexicon is downloaded (Run this once)
# ---------------------------------------------------------
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

# Load the file, preserving "N/A"
filename = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Campo_San_Lorenzo\Campo_San_Lorenzo.csv"
df = pd.read_csv(filename, keep_default_na=False)

# Ensure empty cells are "N/A"
df = df.replace(r'^\s*$', 'N/A', regex=True)

# Define Helper to gather text from row
def get_clean_text(row):
    parts = []
    # Combine relevant columns
    for col in ['title', 'description', 'comments', 'tags']:
        val = str(row[col])
        if val.strip() != 'N/A':
            parts.append(val)
    return " ".join(parts)

# ---------------------------------------------------------
# 1. TextBlob Analysis
# ---------------------------------------------------------
def get_textblob_score(row):
    text = get_clean_text(row)
    if not text:
        return 0.0
    return TextBlob(text).sentiment.polarity

# ---------------------------------------------------------
# 2. VADER Analysis
# ---------------------------------------------------------
sid = SentimentIntensityAnalyzer()

def get_vader_score(row):
    text = get_clean_text(row)
    if not text:
        return 0.0
    # Returns compound score: -1 (Negative) to +1 (Positive)
    return sid.polarity_scores(text)['compound']

# Apply both functions
df['TextBlob_Score'] = df.apply(get_textblob_score, axis=1)
df['Vader_Score'] = df.apply(get_vader_score, axis=1)

output = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Campo_San_Lorenzo\Campo_San_Lorenzo_Both_Scores.csv"
# Save to new CSV
df.to_csv(output, index=False)

# Verify
print(df[['title', 'TextBlob_Score', 'Vader_Score']].head())

                                            title  TextBlob_Score  Vader_Score
0                QUINTESSENZA VENEZIANA 2025 0598           0.000       0.0000
1  0082 - Venice 2023 - IMG_7891_-denoise-sharpen           0.000       0.0000
2  0081 - Venice 2023 - IMG_7891_-denoise-sharpen           0.000       0.0000
3                               campo San Lorenzo           0.000       0.0000
4                 QUINTESSENZA VENEZIANA 2023 662           0.675       0.7901


In [3]:
pip install torch transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
import io

# ---------------------------------------------------------
# DEPENDENCY CHECK & SAFE IMPORT
# ---------------------------------------------------------
# We attempt to import libraries inside try-except blocks.
# If they fail, we add them to a list and exit gracefully with instructions.

missing_libraries = []

try:
    import pandas as pd
except ImportError:
    missing_libraries.append("pandas")

try:
    import requests
except ImportError:
    missing_libraries.append("requests")

try:
    from PIL import Image
    from io import BytesIO
except ImportError:
    missing_libraries.append("pillow")

try:
    import torch
except ImportError:
    missing_libraries.append("torch")

try:
    from transformers import CLIPProcessor, CLIPModel
except ImportError:
    missing_libraries.append("transformers")

if missing_libraries:
    print("---------------------------------------------------------")
    print("⚠️  MISSING LIBRARIES DETECTED")
    print("---------------------------------------------------------")
    print(f"The following libraries are missing: {', '.join(missing_libraries)}")
    print("\nPlease run this command in your terminal/command prompt:")
    print(f"\n    pip install {' '.join(missing_libraries)}\n")
    print("---------------------------------------------------------")
    sys.exit(0)

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
INPUT_FILE = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Campo_San_Lorenzo\Campo_San_Lorenzo_Both_Scores.csv"
OUTPUT_FILE = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Campo_San_Lorenzo\Campo_San_Lorenzo_Sentiment_CLIP.csv"
MODEL_ID = "openai/clip-vit-base-patch32"

# ---------------------------------------------------------
# VENICE SENTIMENT PROMPTS
# CLIP compares the image to these text descriptions.
# ---------------------------------------------------------
POSITIVE_PROMPTS = [
    "a beautiful sunny day in Venice",
    "a romantic gondola ride",
    "peaceful canal water",
    "historic italian architecture",
    "happy people enjoying food",
    "colorful buildings in Venice",
    "artistic and aesthetic photo"
]

NEGATIVE_PROMPTS = [
    "a crowded and stressful tourist trap",
    "garbage and trash on the ground",
    "gloomy grey weather",
    "dirty water",
    "construction work and scaffolding",
    "blurry low quality photo",
    "flooded street acqua alta"
]

ALL_PROMPTS = POSITIVE_PROMPTS + NEGATIVE_PROMPTS

# ---------------------------------------------------------
# INITIALIZE MODEL
# ---------------------------------------------------------
print(f"Loading CLIP model ({MODEL_ID})...")
try:
    model = CLIPModel.from_pretrained(MODEL_ID)
    processor = CLIPProcessor.from_pretrained(MODEL_ID)
except Exception as e:
    print(f"Error loading model: {e}")
    sys.exit(1)

def analyze_image_clip(url):
    """
    Downloads image and compares it against positive/negative text prompts.
    Returns:
        - Most likely description
        - Sentiment Score (-1.0 to 1.0)
    """
    if str(url).strip().lower() == 'n/a' or str(url).strip() == '':
        return "N/A", 0.0

    try:
        # 1. Download Image
        response = requests.get(url, stream=True, timeout=5)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))
        
        # 2. Process inputs
        inputs = processor(
            text=ALL_PROMPTS, 
            images=image, 
            return_tensors="pt", 
            padding=True
        )

        # 3. Run Inference
        with torch.no_grad():
            outputs = model(**inputs)
            
        # 4. Calculate Probabilities
        # logits_per_image: [1, len(ALL_PROMPTS)]
        probs = outputs.logits_per_image.softmax(dim=1)
        probs_list = probs[0].tolist() # Convert to standard python list

        # 5. Calculate Sentiment Score
        # Sum of probabilities for positive prompts minus sum for negative prompts
        # Result will be naturally between -1 and 1
        pos_score = sum(probs_list[:len(POSITIVE_PROMPTS)])
        neg_score = sum(probs_list[len(POSITIVE_PROMPTS):])
        
        final_sentiment = pos_score - neg_score
        
        # Get the single highest matching text description
        max_prob_index = probs_list.index(max(probs_list))
        best_description = ALL_PROMPTS[max_prob_index]

        return best_description, round(final_sentiment, 4)

    except Exception as e:
        # print(f"Error: {e}") 
        return "Error", 0.0

# ---------------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------------
print(f"Reading {INPUT_FILE}...")
try:
    df = pd.read_csv(INPUT_FILE, keep_default_na=False)
except FileNotFoundError:
    print(f"❌ Error: {INPUT_FILE} not found.")
    print("Please ensure the CSV file is in the same directory.")
    sys.exit(1)

df = df.replace(r'^\s*$', 'N/A', regex=True)

clip_desc_list = []
clip_score_list = []

total_rows = len(df)
print(f"Processing {total_rows} images using CLIP (this may be slower than YOLO)...")

for index, row in df.iterrows():
    url = row.get('url_s', 'N/A')
    
    if index % 5 == 0:
        print(f"Processing row {index}/{total_rows}...")
        
    desc, score = analyze_image_clip(url)
    
    clip_desc_list.append(desc)
    clip_score_list.append(score)

# Add new columns
df['CLIP_Best_Desc'] = clip_desc_list
df['CLIP_Visual_Score'] = clip_score_list

# Save
df.to_csv(OUTPUT_FILE, index=False)
print("---------------------------------------------------------")
print(f"Analysis Complete. Saved to {OUTPUT_FILE}")
try:
    print(df[['title', 'CLIP_Best_Desc', 'CLIP_Visual_Score']].head())
except KeyError:
    pass

Loading CLIP model (openai/clip-vit-base-patch32)...
Reading C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Campo_San_Lorenzo\Campo_San_Lorenzo_Both_Scores.csv...
Processing 70 images using CLIP (this may be slower than YOLO)...
Processing row 0/70...
Processing row 5/70...
Processing row 10/70...
Processing row 15/70...
Processing row 20/70...
Processing row 25/70...
Processing row 30/70...
Processing row 35/70...
Processing row 40/70...
Processing row 45/70...
Processing row 50/70...
Processing row 55/70...
Processing row 60/70...
Processing row 65/70...
---------------------------------------------------------
Analysis Complete. Saved to C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Campo_San_Lorenzo\Campo_San_Lorenzo_Sentiment_CLIP.csv
                                            title  \
0                QUINTESSENZA VENEZIANA 2025 0598   
1  0082 - Venice 2023 - IMG_7891_-denoise-sharpen   
2  0081 - Venice 2023 - IMG_7891_-denois

In [4]:
import pandas as pd
import folium
from folium import plugins

# ----------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------
input = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Campo_San_Lorenzo\Campo_San_Lorenzo_Sentiment_CLIP.csv"
df = pd.read_csv(input)

# Make sure sentiment values are floats
df["TextBlob_Score"] = df["TextBlob_Score"].astype(float)
df["Vader_Score"] = df["Vader_Score"].astype(float)
df["CLIP_Visual_Score"] = df["CLIP_Visual_Score"].astype(float)

# ----------------------------------------------------------
# 2. COLOR FUNCTION: -1 (red) → 0 (yellow) → 1 (green)
# ----------------------------------------------------------
def sentiment_to_color(value):
    """
    Converts sentiment score (-1 to 1) into a red-yellow-green gradient.
    """
    # Normalize to 0 → 1
    norm = (value + 1) / 2  

    # Red → Yellow → Green
    r = int(255 * (1 - norm))
    g = int(255 * norm)
    b = 0

    return f"#{r:02x}{g:02x}{b:02x}"

# ----------------------------------------------------------
# 3. BASE MAP SETTINGS
# ----------------------------------------------------------
centre_latitude = 45.437164
centre_longitude = 12.344833

def create_base_map():
    return folium.Map(
        location=[centre_latitude, centre_longitude],
        tiles="Cartodb dark_matter",
        zoom_start=19,
        control_scale=True
    )

# ----------------------------------------------------------
# 4. FUNCTION TO DRAW SENTIMENT MAP
# ----------------------------------------------------------
def build_sentiment_map(df, sentiment_column, file_output):
    m = create_base_map()

    for _, row in df.iterrows():
        value = row[sentiment_column]
        color = sentiment_to_color(value)

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8
        ).add_to(m)

    m.save(file_output)
    print(f"Saved: {file_output}")

# ----------------------------------------------------------
# 5. GENERATE THREE MAPS
# ----------------------------------------------------------
build_sentiment_map(df, "TextBlob_Score", "map_TextBlob.html")
build_sentiment_map(df, "Vader_Score", "map_Vader.html")
build_sentiment_map(df, "CLIP_Visual_Score", "map_CLIP.html")


Saved: map_TextBlob.html
Saved: map_Vader.html
Saved: map_CLIP.html


In [6]:
import pandas as pd
import folium
from folium import plugins

# ----------------------------------------------------------
# 1. LOAD + CLEAN DATA
# ----------------------------------------------------------
input = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Giardini_della_Marinaressa\Giardini_della_Marinaressa_Sentiment_CLIP.csv"
df = pd.read_csv(input)

# Convert everything to numeric (invalid values become NaN)
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["TextBlob_Score"] = pd.to_numeric(df["TextBlob_Score"], errors="coerce")
df["Vader_Score"] = pd.to_numeric(df["Vader_Score"], errors="coerce")
df["CLIP_Visual_Score"] = pd.to_numeric(df["CLIP_Visual_Score"], errors="coerce")

# Remove rows with missing important values
df = df.dropna(subset=["latitude", "longitude",
                       "TextBlob_Score", "Vader_Score", "CLIP_Visual_Score"])

print("Cleaned dataset size:", len(df))

# ----------------------------------------------------------
# 2. COLOR FUNCTION (-1 → +1) = (red → green)
# ----------------------------------------------------------
def sentiment_to_color(value):
    if pd.isna(value):
        return "#808080"  # fallback grey (should not happen now)
    
    norm = (value + 1) / 2  # convert -1..1 to 0..1

    r = int(255 * (1 - norm))
    g = int(255 * norm)
    
    return f"#{r:02x}{g:02x}00"

# ----------------------------------------------------------
# 3. BASE MAP SETTINGS
# ----------------------------------------------------------
centre_latitude = 45.431488
centre_longitude = 12.352647

def create_base_map():
    return folium.Map(
        location=[centre_latitude, centre_longitude],
        tiles="Cartodb dark_matter",
        zoom_start=19,
        control_scale=True
    )

# ----------------------------------------------------------
# 4. FUNCTION TO DRAW SENTIMENT MAP
# ----------------------------------------------------------
def build_sentiment_map(df, sentiment_column, file_output):
    m = create_base_map()

    for _, row in df.iterrows():
        value = row[sentiment_column]
        color = sentiment_to_color(value)

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8
        ).add_to(m)

    m.save(file_output)
    print(f"Saved: {file_output}")

# ----------------------------------------------------------
# 5. GENERATE THREE MAPS
# ----------------------------------------------------------
build_sentiment_map(df, "TextBlob_Score", "map_TextBlob.html")
build_sentiment_map(df, "Vader_Score", "map_Vader.html")
build_sentiment_map(df, "CLIP_Visual_Score", "map_CLIP.html")


Cleaned dataset size: 356
Saved: map_TextBlob.html
Saved: map_Vader.html
Saved: map_CLIP.html
